In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict

# =============================================================================
# 0) Configuration block (2x2 grid)
# =============================================================================
PLOT_CFG = {
    "height": 2.5,
    "aspect": 1.5,
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",
    "legend_fontsize": 9,
    "outfile": "result_single_vs_repeated_2x2.pdf",
}

def make_title_map(models):
    """Map model names to SINGLE / REPEATED using prefix matching."""
    d = {}
    for m in models:
        if m.startswith("sec31_base"):
            d[m] = "SINGLE"
        elif m.startswith("sec31_multi"):
            d[m] = "REPEATED"
    return d

METRIC_MAP: Dict[str, str] = {
    'em/pku':      r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/icku':     r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pref_pk':  r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pref_ick': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}

LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (1, 1),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (1, 1),
}

# =============================================================================
# 1) Style setup
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        },
    )

# =============================================================================
# 2) Data loading & statistics
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    title_map = make_title_map(df["model"].unique())

    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    long = pd.melt(
        df, id_vars=["model", "step"], value_vars=metrics,
        var_name="Metric", value_name="Accuracy",
    )
    if long["Accuracy"].max() <= 1.0:
        long["Accuracy"] *= 100.0

    long["Model_Title"] = long["model"].map(title_map).fillna(long["model"])
    long["Mean"] = long["Accuracy"]
    long["Std"] = 0.0

    return (
        long[["model", "Model_Title", "step", "Metric", "Mean", "Std"]]
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )

# =============================================================================
# 3) 2x2 FacetGrid plotting
# =============================================================================
def plot_2x2_facet(stats_df: pd.DataFrame) -> None:
    acc_metrics = [r'$\mathrm{Acc}_{\mathrm{PKU}}$', r'$\mathrm{Acc}_{\mathrm{ICKU}}$']
    pref_metrics = [r'$\mathrm{Pref}_{\mathrm{PK}}$', r'$\mathrm{Pref}_{\mathrm{ICK}}$']

    df = stats_df.copy()
    df["MetricType"] = df["Metric"].apply(
        lambda x: "Acc" if x in acc_metrics else ("Pref" if x in pref_metrics else None)
    )
    df = df.dropna(subset=["MetricType"])
    df = df[df["Model_Title"].isin(["SINGLE", "REPEATED"])]

    g = sns.FacetGrid(
        df, row="MetricType", col="Model_Title",
        row_order=["Acc", "Pref"], col_order=["SINGLE", "REPEATED"],
        sharey="row", sharex=True,
        height=PLOT_CFG["height"], aspect=PLOT_CFG["aspect"], margin_titles=False,
    )

    def facet_plot(data, **kwargs):
        ax = plt.gca()
        for metric, mdf in data.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            ax.plot(mdf["step"], mdf["Mean"], label=metric,
                    color=COLOR_MAP[metric], dashes=LINESTYLE_MAP[metric])
            ax.fill_between(mdf["step"], mdf["Mean"] - mdf["Std"], mdf["Mean"] + mdf["Std"],
                            color=COLOR_MAP[metric], alpha=PLOT_CFG["shade_alpha"])

    g.map_dataframe(facet_plot)

    for ax in g.axes.flat:
        ax.grid(alpha=0.3)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(handles, labels, fontsize=PLOT_CFG["legend_fontsize"], loc="center right", frameon=True)

    for ax, title in zip(g.axes[0], ["SINGLE", "REPEATED"]):
        ax.set_title(title, fontsize=13, weight="bold", style="italic")
    for ax in g.axes[1]:
        ax.set_title("")
    for ax in g.axes[1]:
        ax.set_xlabel(PLOT_CFG["xlabel"])
    for ax in g.axes[:, 0]:
        ax.set_ylabel(PLOT_CFG["ylabel"])

    g.fig.tight_layout()
    g.fig.savefig(PLOT_CFG["outfile"], bbox_inches="tight", format="pdf")
    print(f"Saved 2x2 FacetGrid plot as {PLOT_CFG['outfile']}")
    plt.show()

# =============================================================================
# 4) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    df_stats = load_stats("../probe_results.csv")
    plot_2x2_facet(df_stats)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict

PLOT_CFG = {
    "height": 2.0, "aspect": 1.7, "linewidth": 2.0, "shade_alpha": 0.18,
    "style": "whitegrid", "palette": "deep", "dpi": 150,
    "xlabel": "Training Steps", "ylabel": "Accuracy (%)",
    "legend_fontsize": 10, "outfile": "result_single_vs_repeated.pdf",
}

def make_title_map(models):
    d = {}
    for m in models:
        if m.startswith("sec31_base"):  d[m] = "SINGLE"
        elif m.startswith("sec31_multi"): d[m] = "REPEATED"
    return d

METRIC_MAP: Dict[str, str] = {
    'em/pku':      r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/icku':     r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pref_pk':  r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pref_ick': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}

LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (1, 1),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (1, 1),
}

PANEL_TITLE_MAP = {"SINGLE": "SINGLE", "REPEATED_ACC": "REPEATED", "REPEATED_PREF": "REPEATED"}

def setup_style():
    sns.set_theme(style=PLOT_CFG["style"], palette=PLOT_CFG["palette"], rc={
        "figure.dpi": PLOT_CFG["dpi"], "axes.titlesize": 14, "axes.labelsize": 12,
        "legend.fontsize": PLOT_CFG["legend_fontsize"], "lines.linewidth": PLOT_CFG["linewidth"],
    })

def load_stats(csv_path):
    df = pd.read_csv(csv_path)
    title_map = make_title_map(df["model"].unique())
    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())
    long = pd.melt(df, id_vars=["model", "step"], value_vars=metrics, var_name="Metric", value_name="Accuracy")
    if long["Accuracy"].max() <= 1.0: long["Accuracy"] *= 100.0
    long["Model_Title"] = long["model"].map(title_map).fillna(long["model"])
    long["Mean"] = long["Accuracy"]; long["Std"] = 0.0
    return long[["model", "Model_Title", "step", "Metric", "Mean", "Std"]].sort_values(["Model_Title", "Metric", "step"]).reset_index(drop=True)

def add_panel_column(stats_df):
    df = stats_df.copy(); df["Panel"] = None
    acc = [r'$\mathrm{Acc}_{\mathrm{PKU}}$', r'$\mathrm{Acc}_{\mathrm{ICKU}}$']
    pref = [r'$\mathrm{Pref}_{\mathrm{PK}}$', r'$\mathrm{Pref}_{\mathrm{ICK}}$']
    df.loc[(df["Model_Title"]=="SINGLE") & df["Metric"].isin(acc), "Panel"] = "SINGLE"
    df.loc[(df["Model_Title"]=="REPEATED") & df["Metric"].isin(acc), "Panel"] = "REPEATED_ACC"
    df.loc[(df["Model_Title"]=="REPEATED") & df["Metric"].isin(pref), "Panel"] = "REPEATED_PREF"
    return df.dropna(subset=["Panel"])

def plot_three_facet(stats_df):
    data = add_panel_column(stats_df)
    g = sns.FacetGrid(data, col="Panel", col_order=["SINGLE", "REPEATED_ACC", "REPEATED_PREF"],
                      sharey=True, height=PLOT_CFG["height"], aspect=PLOT_CFG["aspect"], margin_titles=False)
    def facet_plot(data, **kwargs):
        ax = plt.gca()
        for metric, mdf in data.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            ax.plot(mdf["step"], mdf["Mean"], label=metric, color=COLOR_MAP[metric], dashes=LINESTYLE_MAP[metric])
            ax.fill_between(mdf["step"], mdf["Mean"]-mdf["Std"], mdf["Mean"]+mdf["Std"], color=COLOR_MAP[metric], alpha=PLOT_CFG["shade_alpha"])
    g.map_dataframe(facet_plot)
    for ax, pn in zip(g.axes.flat, ["SINGLE", "REPEATED_ACC", "REPEATED_PREF"]):
        ax.set_title(PANEL_TITLE_MAP[pn], fontsize=13, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"]); ax.grid(alpha=0.3)
        h, l = ax.get_legend_handles_labels()
        ax.legend(h, l, fontsize=PLOT_CFG["legend_fontsize"], loc="center right", frameon=True)
    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"])
    g.fig.tight_layout()
    g.fig.savefig(PLOT_CFG["outfile"], bbox_inches="tight", format="pdf")
    print(f"Saved FacetGrid plot as {PLOT_CFG['outfile']}"); plt.show()

if __name__ == "__main__":
    setup_style(); plot_three_facet(load_stats("../probe_results.csv"))

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List

PLOT_CFG = {
    "height": 2.0, "aspect": 1.7, "linewidth": 2.0, "shade_alpha": 0.18,
    "style": "whitegrid", "palette": "deep", "dpi": 150,
    "xlabel": "Training Steps", "ylabel": "Accuracy (%)",
    "legend_out": True, "legend_loc": "center left", "legend_bbox": (1.02, 0.5),
    "legend_ncol": 1, "legend_fontsize": 11, "right_pad": 1,
    "outfile": "result_noise_1percent.pdf",
}

def make_title_map(models):
    d = {}
    for m in models:
        if m.startswith("sec31_multi"):    d[m] = "without noise"
        elif m.startswith("sec32_noise001"): d[m] = "1% noise"
    return d

METRIC_MAP: Dict[str, str] = {
    'em/pku':      r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/icku':     r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pref_pk':  r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pref_ick': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}

LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (2, 4),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (2, 4),
}

def setup_style():
    sns.set_theme(style=PLOT_CFG["style"], palette=PLOT_CFG["palette"], rc={
        "figure.dpi": PLOT_CFG["dpi"], "axes.titlesize": 14, "axes.labelsize": 12,
        "legend.fontsize": PLOT_CFG["legend_fontsize"], "lines.linewidth": PLOT_CFG["linewidth"],
    })

def load_stats(csv_path):
    df = pd.read_csv(csv_path)
    title_map = make_title_map(df["model"].unique())
    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())
    long = pd.melt(df, id_vars=["model", "step"], value_vars=metrics, var_name="Metric", value_name="Accuracy")
    if long["Accuracy"].max() <= 1.0: long["Accuracy"] *= 100.0
    long["Model_Title"] = long["model"].map(title_map).fillna(long["model"])
    long["Mean"] = long["Accuracy"]; long["Std"] = 0.0
    return long[["model", "Model_Title", "step", "Metric", "Mean", "Std"]].sort_values(["Model_Title", "Metric", "step"]).reset_index(drop=True)

def plot_with_std(stats_df, model_titles, outfile=None):
    data = stats_df[stats_df["Model_Title"].isin(model_titles)].copy()
    if data.empty: print("No data"); return
    g = sns.FacetGrid(data, col="Model_Title", col_order=model_titles,
                      height=PLOT_CFG["height"], aspect=PLOT_CFG["aspect"], sharey=True, margin_titles=False)
    for ax, title in zip(g.axes.flat, model_titles):
        sub = data[data["Model_Title"] == title]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            ax.plot(mdf["step"], mdf["Mean"], label=metric, color=COLOR_MAP[metric], dashes=LINESTYLE_MAP[metric])
            ax.fill_between(mdf["step"], mdf["Mean"]-mdf["Std"], mdf["Mean"]+mdf["Std"], color=COLOR_MAP[metric], alpha=PLOT_CFG["shade_alpha"])
        ax.set_title(title, fontsize=14, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"]); ax.grid(True, alpha=0.3)
    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"])
    if PLOT_CFG["legend_out"]:
        h, l = g.axes.flat[0].get_legend_handles_labels()
        g.fig.legend(h, l, loc=PLOT_CFG["legend_loc"], bbox_to_anchor=PLOT_CFG["legend_bbox"],
                     ncol=PLOT_CFG["legend_ncol"], fontsize=PLOT_CFG["legend_fontsize"], frameon=True)
        g.fig.subplots_adjust(right=PLOT_CFG["right_pad"])
    for ax in g.axes.flat:
        leg = ax.get_legend()
        if leg: leg.remove()
    out = outfile or PLOT_CFG["outfile"]
    g.fig.savefig(out, dpi=PLOT_CFG["dpi"], bbox_inches="tight")
    print(f"Saved: {out}"); plt.show()

if __name__ == "__main__":
    setup_style()
    plot_with_std(load_stats("../probe_results.csv"), ["1% noise"])

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

CSV_PATH   = "../probe_results.csv"
METRIC_COL = "em/icku"   # Acc_ICKU

def find_model_by_prefix(df, prefix):
    matches = [m for m in df["model"].unique() if m.startswith(prefix)]
    return matches[0] if matches else None

df = pd.read_csv(CSV_PATH)

model_0pct  = find_model_by_prefix(df, "sec31_multi")
model_1pct  = find_model_by_prefix(df, "sec32_noise001")
model_5pct  = find_model_by_prefix(df, "sec32_noise005")
model_10pct = find_model_by_prefix(df, "sec32_noise010")

MODEL_IDS = [m for m in [model_0pct, model_1pct, model_5pct, model_10pct] if m is not None]
MODEL_LABELS = {}
if model_0pct:  MODEL_LABELS[model_0pct]  = "0%"
if model_1pct:  MODEL_LABELS[model_1pct]  = "1%"
if model_5pct:  MODEL_LABELS[model_5pct]  = "5%"
if model_10pct: MODEL_LABELS[model_10pct] = "10%"
ORDER = ["0%", "1%", "5%", "10%"]

sns.set_theme(style="whitegrid", rc={"figure.dpi": 150, "axes.titlesize": 13, "axes.labelsize": 12})

if METRIC_COL in df.columns and df[METRIC_COL].max() <= 1.0:
    df[METRIC_COL] *= 100.0

rows = []
for mid in MODEL_IDS:
    sub = df[df["model"] == mid].copy()
    if sub.empty: continue
    last_step = sub["step"].max()
    sub_last = sub[sub["step"] == last_step][["model", METRIC_COL]].copy()
    sub_last["label"] = MODEL_LABELS.get(mid, mid)
    rows.append(sub_last)

last_df = pd.concat(rows, ignore_index=True).dropna(subset=[METRIC_COL]) if rows else pd.DataFrame()
if not last_df.empty:
    last_df["label"] = pd.Categorical(last_df["label"], categories=ORDER, ordered=True)
    print(last_df.groupby("label", as_index=False)[METRIC_COL].agg(mean="mean", n="count").round(2))

fig, ax = plt.subplots(figsize=(2.5, 2.5))
sns.barplot(data=last_df, x="label", y=METRIC_COL, order=ORDER,
            estimator=np.mean, errorbar=None, width=0.7, ax=ax)
ax.set_xlabel("Noise"); ax.set_ylabel(r'$\mathrm{Acc}_{\mathrm{ICKU}}$' + "(%)")
ax.grid(True, axis="y", linestyle="--", alpha=0.5); sns.despine()
plt.tight_layout(); plt.savefig("bars_acc_icku_last.pdf", bbox_inches="tight", dpi=300); plt.show()

In [ ]:
# =============================================================================
# Per-entity probability analysis: Binned prob vs Pref_PK (all attributes)
# SKIPPED: Requires fullprob_*.csv files that have not been generated yet
# for the current project. Re-enable this cell once those files are available.
# =============================================================================
print("SKIPPED: This cell requires fullprob_*.csv data not yet generated.")

In [ ]:
# =============================================================================
# Post-training prob vs Pref_PK analysis (Scenario 1 & 2)
# SKIPPED: Requires fullprob_*.csv files that have not been generated yet
# for the current project. Re-enable this cell once those files are available.
# =============================================================================
print("SKIPPED: This cell requires fullprob_*.csv data not yet generated.")